ANALISI NCT considerando transizioni di stato all'interno di ciascuna fase:
- pre-ictale:   x0 [0, 10]s -> xf [110, 120]s
- ictale:   x0 [marker1=180, marker1+10=190]s -> xf [marker2-10=223.5, marker2=233.5]s

In [1]:
# ## IMPORT LIBRERIE (paper) & PATH BASE
# --- standard/core ---
import os
from pathlib import Path
import re
import numpy as np
import pandas as pd

import scipy as sp
from scipy.io import loadmat
from scipy import stats
from scipy.spatial import distance
from sklearn.cluster import KMeans
from tqdm import tqdm

# opzionali / velocità / grafica: scikit-learn per KDTree 
# (altrimenti fallback NumPy)
try:
    from sklearn.neighbors import KDTree
    HAVE_SKLEARN = True
except Exception:
    HAVE_SKLEARN = False

# --- plotting ---
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams.update({"font.size": 10})
plt.rcParams["svg.fonttype"] = "none"
try:
    import seaborn as sns
except Exception:
    sns = None

# --- neuroimaging ---
try:
    from nilearn import datasets, plotting  # richiede nibabel
except Exception:
    datasets = plotting = None
# carica atlanti/surface e plot cerebrali.

# --- nctpy (dal toolkit del paper, Procedure 1) ---
from nctpy.energies import integrate_u, get_control_inputs
from nctpy.pipelines import ComputeControlEnergy, ComputeOptimizedControlEnergy
from nctpy.metrics import ave_control
from nctpy.utils import (
    matrix_normalization,
    convert_states_str2int,
    normalize_state,
    normalize_weights,
    get_null_p,
    get_fdr_p,
)

from nctpy.plotting import (
    roi_to_vtx,
    null_plot,
    surface_plot,
    add_module_lines,  
)
# ATTENZIONE: null models è un modulo interno al repository degli autori, non un pacchetto PyPI “standard”. 
# Posso scegliere un strada alternativa richiamando surrogati equivalenti da librerie note.

# =============================
# PATH FILE & DATASETS
# =============================
try:
    ROOT = Path(__file__).resolve().parent
except NameError:
    # Notebook / VS Code Interactive
    try:
        from IPython import get_ipython
        ip = get_ipython()
        # current working dir of the kernel/session
        ROOT = Path(ip.run_line_magic('pwd', ''))
    except Exception:
        ROOT = Path.cwd()


BASE = ROOT / "NCT_analysis\data"

B0_TXT    = BASE / "centroidsB0_850ROIs.txt"
EPI_TXT   = BASE / "centroidsEPI_850ROIs.txt"
MNI_XLSX  = BASE / "centroidsMNI2mm_850ROIs.xlsx"
SEEG_MONO = BASE / "sEEG_3D_contacts_locs.mat"
SEEG_BI   = BASE / "sEEG_3D_BIPOLARcontacts_locs.mat"

# Cartella per gli output
OUT = ROOT / "within_phases_analysis"
OUT.mkdir(exist_ok=True)

K_NEAREST = 1  # numero di vicini più vicini da considerare per l’estrazione del control set

CTL_TXT   = ROOT / "output_definitivo" / f"control_set_ROIs_B0_BI_top1_perContact_k{K_NEAREST}_LH.txt"   #  control set
CONN_XLSX = BASE / "connectome850_2b0.xlsx"                    #  connettoma strutturale


CONTROL ENERGY WITHIN PRE-ICTAL PHASE

In [2]:
# -- Caricamento CONTROL SET (lista di indici 1-based, con duplicati e ordinati) --
try:
    ctl_ids = np.loadtxt(CTL_TXT, dtype=int, ndmin=1)  # mantiene duplicati e ordine
except Exception:
    # fallback robusto se il file avesse righe/spazi strani
    ctl_ids = []
    with open(CTL_TXT, "r") as f:
        for line in f:
            line = line.strip()
            if line:
                ctl_ids.append(int(float(line)))
    ctl_ids = np.array(ctl_ids, dtype=int)

print(f"[CHECK] Control set (BI per-contatto): letti {ctl_ids.size} indici da {CTL_TXT.name}")
if ctl_ids.size != 77:
    print("[WARNING] Il control set non ha 77 righe. Controlla l’estrazione per-contatto e MAX_DIST=None.")


# VINCOLO EMISFERO SINISTRO (ID ammessi)
ALLOWED_LH_IDS = set(list(range(1, 401))+list(range(826, 851)))  # (Shaefer + Tian) solo LH
bad = sorted(set(ctl_ids.tolist()) - ALLOWED_LH_IDS)
if bad:
    raise ValueError(f"[CHECK] Control set NON LH-only: trovati ID fuori set consentito: {bad[:20]}")
print("[CHECK] Control set LH-only ✅")

# -- Caricamento CONNETTOMA da Excel→ numpy, fix semplice se non quadrato --
def load_adjacency_from_excel(path: Path) -> np.ndarray:
    df = pd.read_excel(path, header=None)
    # Forza numerico, elimina colonne/righe completamente vuote
    for c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df.dropna(axis=0, how="all").dropna(axis=1, how="all")
    A = df.to_numpy(dtype=float)

    # Se non quadrata, tenta una correzione semplice (drop prima riga/col)
    if A.shape[0] != A.shape[1]:
        # prova drop col0
        if A.shape[1] - 1 == A.shape[0]:
            A = A[:, 1:]
        # prova drop row0
        elif A.shape[0] - 1 == A.shape[1]:
            A = A[1:, :]
        # prova drop row0 e col0
        elif A.shape[0] - 1 == A.shape[1] - 1:
            A = A[1:, 1:]
    assert A.shape[0] == A.shape[1], f"Matrice non quadrata anche dopo fix: {A.shape}"
    # Sostituisci eventuali NaN con 0
    A = np.nan_to_num(A, copy=False)
    return A

A = load_adjacency_from_excel(CONN_XLSX)
N = A.shape[0]  #n_nodes
print(f"[CHECK] Connettoma caricato: shape={A.shape}")
# -- PARCELLATION: 800 vs 850 e coerenza con control set --
if N == 850:
    print("[CHECK] Parcellazione = 850 ✅")
elif N == 800:
    print("[CHECK] Parcellazione = 800 ⚠️ verifica control set")
else:
    print(f"[CHECK] Parcellazione non standard (N={N}) ⚠️")
# range e duplicati nel control set
if ctl_ids.min() < 1 or ctl_ids.max() > N:
    raise ValueError(f"[CHECK] ID control set fuori range 1..{N} (min={ctl_ids.min()}, max={ctl_ids.max()}).")

# pulizia A: simmetria e diagonale
if not np.isfinite(A).all():
    raise ValueError("[CHECK] A contiene NaN/±inf.")
# simmetria (per reti non dirette ci si aspetta A≈A^T)
asym = np.linalg.norm(A - A.T, ord='fro') / max(1.0, np.linalg.norm(A, ord='fro'))
if asym > 1e-6:
    print(f"[CHECK] A non perfettamente simmetrica (relFro={asym:.2e}) → simmetrizzo.")
    A = (A + A.T) / 2.0
# diagonale: molte pipeline NCT usano diag=0 (niente self-loops)
n_diag_nz = int(np.sum(np.abs(np.diag(A)) > 1e-12))
if n_diag_nz > 0:
    print(f"[CHECK] Diagonale con {n_diag_nz} valori ≠0 → azzero.")
    np.fill_diagonal(A, 0.0)

# -- Salvataggi “puliti” e report sintetico --
pd.DataFrame(A).to_csv(OUT / f"A_struct_clean_LHonly_k{K_NEAREST}.csv", index=False, header=False)
pd.DataFrame([{
    "N": N,
    "control_set_size": int(ctl_ids.size),
    "control_set_min": int(ctl_ids.min()),
    "control_set_max": int(ctl_ids.max()),
    "diag_nonzero_before": n_diag_nz,
    "asymmetry_relFro": float(asym),
}]).to_csv(OUT / f"control_energy_qc_summary_LHonly_k{K_NEAREST}.csv", index=False)

# -- Preparazione vettore b e matrice B per la NCT --
# b: vettore di controllo (1 se ROI controllata, 0 altrimenti), 1-based → 0-based
b = np.zeros(N, dtype=float)
b[ctl_ids - 1] = 1.0    # le righe duplicate non cambiano il risultato (assegnazioni ripetute a 1)
B = np.diag(b)
print("rank(B) = ", np.linalg.matrix_rank(B))
pd.DataFrame(B).to_csv(OUT / f"B_diag_matrix_LHonly_k{K_NEAREST}.csv", index=False, header=False)
# con ciò, u(t) può essere applicato solo sui controller, quindi l'energia 
# sarà concentrata su di essi per costruzione.


[CHECK] Control set (BI per-contatto): letti 77 indici da control_set_ROIs_B0_BI_top1_perContact_k1_LH.txt
[CHECK] Control set LH-only ✅
[CHECK] Connettoma caricato: shape=(850, 850)
[CHECK] Parcellazione = 850 ✅
[CHECK] Diagonale con 833 valori ≠0 → azzero.
rank(B) =  43


In [5]:
# ============================ Time System — passo 1 =========================================
# system = "discrete" # or system = "continuous" --- come nel paper: continuo ---
system = "continuous"

# ============================ Normalizzazione di A — passo 2 =========================================
# 2. Normalize the adjacency matrix. 
A_norm = matrix_normalization(A, system=system, c=1)
print("A_norm pronto (system=continuous, c=1).")

# ========================= Definizione degli stati del control task — passo 3 ============================
# 3. Definizione degli stati iniziale e target (x0, xf) dal control set
def load_roi_list_1based_to_mask(txt_path: Path, n_nodes: int) -> np.ndarray:
    ids = []
    with open(txt_path, "r") as f:
        for line in f:
            parts = line.replace(",", " ").split()
            if parts:
                try:
                    idx1 = int(float(parts[0]))
                    i0 = idx1 - 1
                    if 0 <= i0 < n_nodes:
                        ids.append(i0)
                except ValueError:
                    pass
    ids = sorted(set(ids))
    mask = np.zeros(n_nodes, dtype=bool)
    mask[ids] = True
    return mask

controllers = load_roi_list_1based_to_mask(CTL_TXT, N) # 77 ROI da BI per-contatto
n_ctrl = int(controllers.sum())
print(f"[STATE] #ROI controllate (da BI per-contatto, deduplicate) = {n_ctrl} / {N}")

non_controllers  = ~controllers    # è solo una mask per distinguere nodi controllati vs non controllati
                                # (utile per B e per i plot)
strength = A.sum(axis=0)    # Strength (diagnostica)
print("  controllers' mean strength: {:.2f}".format(np.mean(strength[controllers])))
print("  non-controllers'  mean strength: {:.2f}".format(np.mean(strength[non_controllers])))


# Caricamento da MATLAB degli stati x0 = x_preictal1_80_250 & xf = x_preictal2_80_250
STATE_MATE = BASE / "within_phases_states.mat"
mat_states = loadmat(STATE_MATE)
print(sorted(mat_states.keys()))

x_preictal1 = np.asarray(mat_states["x_preictal1_80_250"], dtype=float).ravel()
x_preictal2 = np.asarray(mat_states["x_preictal2_80_250"], dtype=float).ravel()

print("x_preictal1 shape:", x_preictal1.shape)
print("x_preictal2 shape:", x_preictal2.shape)

assert x_preictal1.shape[0] == N == A.shape[0]
assert x_preictal2.shape[0] == N == A.shape[0]

# --- Normalizzazione come nel paper: stato di norma unitaria ---
initial_state_raw1 = x_preictal1 # stato iniziale grezzo 
target_state_raw1  = x_preictal2 # stato target grezzo 

initial_state1 = normalize_state(initial_state_raw1)
target_state1  = normalize_state(target_state_raw1)

# salvo gli stati usati 
np.save(OUT / "initial_state_850_sEEG_LHonly.npy", np.asarray(initial_state1, dtype=float))
np.save(OUT / "target_state_850_sEEG_LHonly.npy",  np.asarray(target_state1, dtype=float))
print("[STATE] Stati iniziale e target presi da PSD sEEG (non-binari, normalizzati).")

"adesso, initial state = x_preictal1 normalizzato ; target state = x_preictal2 normalizzato;"
"non c'è più alcun complemento: sono proprio i pattern di PSD z-scored normalizzati calcolati in MATLAB"



# == Partial control (1 su miei nodi, 1e-3 sugli altri): con questo scelgo dove applicare i control signals
control_set = np.zeros((N, N), dtype=float) # control_set =np.eye(N) x controllo completo su tutti i nodi (optimal control)
control_set[controllers, controllers] = 1.0 # controllo completo sui nodi controllati
control_set[~controllers, ~controllers] = 1e-3 # controllo minimo sui nodi non controllati (mettere a 0)



# ============================ Calcolo Control Energy — passo 4 ==============================
# Calcolo dei control signals u(t) & state trajectory x(t).
# parameteri e risoluzione 
time_horizon = 1             # come nel paper (unità arbitraria)
rho = 1                      # optimal control (peso uguale a traiettoria e input)
#S = np.diag(initial_mask.astype(float))  # S=trajectory_constraints: vincola solo i nodi controllati
S = np.eye(N)         # S=trajectory_constraints: vincola tutti i nodi (optimal control)


# ottengo la state trajectory x(t) & i control signals u(t)
state_trajectory, control_signals, numerical_error = get_control_inputs(
    A_norm=A_norm,
    T=time_horizon,
    B=control_set,      # solo controller
    x0=initial_state1,
    xf=target_state1,
    system=system,
    rho=rho,
    S=S,               
)

# Check inversion / reconstruction errors
thr = 1e-8  #threshold
print("inversion error = {:.2E} (<{:.2E}={:})".format(numerical_error[0], thr, numerical_error[0] < thr))
print("reconstruction error = {:.2E} (<{:.2E}={:})".format(numerical_error[1], thr, numerical_error[1] < thr))
print("rank(B) = ", np.linalg.matrix_rank(control_set))

xT = state_trajectory[-1]  # stato finale
err_ctrl = np.linalg.norm(xT[controllers] - target_state1[controllers])
print("Final error (controllers):", err_ctrl)    # deve essere ragionevole
drift_non = np.linalg.norm(xT[~controllers] - initial_state1[~controllers])
print("Final drift non-controllers:", drift_non)



# ============================ Plot x(t) & u(t) — passo 5 ============================
# Nota: nctpy ritorna di solito matrici shape (T, N). Il paper usa righe=tempo, colonne=nodi.
mask_init1 = initial_state_raw1 > np.median(initial_state_raw1) # initial = nodi con PSD interictale alta (sopra la mediana del pattern interictale)
mask_target1 = target_state_raw1 > np.median(target_state_raw1) # target = nodi con PSD ictale alta (sopra la mediana del pattern ictale)
mask_byst1 = ~(mask_init1 | mask_target1)  # bystander = tutto il resto
# queste maschere servono solo per suddividere l'energia per gruppi funzionali e colorare i plot.

T_steps = state_trajectory.shape[0]           # es. 1001
t = np.linspace(0, time_horizon, T_steps)     # asse del tempo [0, T]

f, ax = plt.subplots(3, 2, figsize=(7, 7))

# INITIAL NODES x0:
#  A | control signals, x0
ax[0, 0].plot(t, control_signals[:, initial_state1 != 0], linewidth=0.75)
ax[0, 0].set_title("A | control signals, x0")

# B | neural activity, x0
ax[0, 1].plot(t, state_trajectory[:, initial_state1 != 0], linewidth=0.75)
ax[0, 1].set_title("B | neural activity, x0")

# TARGET NODES xf:
# C | control signals, xf
ax[1, 0].plot(t, control_signals[:, target_state1 != 0], linewidth=0.75)
ax[1, 0].set_title("C | control signals, xf")

# D | neural activity, xf
ax[1, 1].plot(t, state_trajectory[:, target_state1 != 0], linewidth=0.75)
ax[1, 1].set_title("D | neural activity, xf")

# BYSTANDER NODES: possono essere controllers o non-controllers
byst = np.logical_and(initial_state1 == 0, target_state1 == 0)
# E | control signals, bystanders
ax[2, 0].plot(t, control_signals[:, byst], linewidth=0.75)
ax[2, 0].set_title("E | control signals, bystanders")

# F | neural activity, bystanders
ax[2, 1].plot(t, state_trajectory[:, byst], linewidth=0.75)
ax[2, 1].set_title("F | neural activity, bystanders")

for cax in ax.reshape(-1):
    cax.set_ylabel("activity")
    cax.set_xlabel("time (a.u.)")
    cax.set_xlim(0, time_horizon)
    cax.set_xticks([0, time_horizon])    
    cax.set_xticklabels([0, time_horizon])
f.tight_layout()
(OUT / "control_energy_PREICTAL").mkdir(exist_ok=True)
f.savefig(OUT / "control_energy_PREICTAL" / f"plot_xu_850_LHonly_k{K_NEAREST}.svg", dpi=600, bbox_inches="tight", pad_inches=0.01)
plt.close(f)

# ============================ Control energy — passo 6 =====================================
# integro i control signals
def integrate_u_compat(u, T):
    """
    Compat fallback per integrare l'energia:
    restituisce il vettore per-nodo ∫ u(t)^2 dt usando Simpson se disponibile
    oppure trapezi come fallback.
    u: array shape (T_steps, N)
    """
    try:
        # prova ad usare nctpy.energies.integrate_u
        return integrate_u(u)
    except Exception:
        # fallback robusto (trapezi): differenze minime con molti step
        
        dt = T / (u.shape[0] - 1)
        return np.trapezoid(u**2, dx=dt, axis=0)

# (A) Calcolo node-level control energy integrando i control signals:
node_energy = integrate_u_compat(control_signals, time_horizon)  # shape (N,)
print("node_energy shape:", node_energy.shape)

E_ctrl = float(node_energy[controllers].sum())
E_non  = float(node_energy[~controllers].sum())
print("Energy controllers:", E_ctrl)
print("Energy non-controllers:", E_non)

# salvataggio
np.save(OUT / "control_energy_PREICTAL"/ f"node_energy_850_LHonly_k{K_NEAREST}.npy", np.asarray(node_energy, dtype=float))
print("[SAVE] node_energy_850_LHonly.npy salvato in", OUT)

# (B) Riassumo nodal energy:
energy = float(np.sum(node_energy))
print("Total control energy:", round(energy, 4))

# ============================ Salvataggi e riepilogo ===================================
RESULT = OUT / "control_energy_PREICTAL"
RESULT.mkdir(exist_ok=True)
pd.DataFrame({"node": np.arange(1, N + 1), "node_energy": node_energy}).to_csv(
    RESULT / f"node_energy_850_LHonly_k{K_NEAREST}.csv", index=False
)
with open(RESULT / f"summary_LHonly_k{K_NEAREST}.txt", "w") as fsum:
    fsum.write(
        f"N={N}\n"
        f"energy={energy:.10f}\n"
        f"inversion_err={numerical_error[0]:.4e}\n"
        f"recon_err={numerical_error[1]:.4e}\n"
        f"control_set_file={CTL_TXT.name}\n"
    )
print(f"[DONE] Risultati scritti in: {RESULT}")



A_norm pronto (system=continuous, c=1).
[STATE] #ROI controllate (da BI per-contatto, deduplicate) = 43 / 850
  controllers' mean strength: 24855.77
  non-controllers'  mean strength: 16561.80
['P_ICT_1', 'P_ICT_2', 'Pband_ICT1_80_250', 'Pband_ICT2_80_250', '__globals__', '__header__', '__version__', 'a_ICT1_80_250', 'a_ICT1_z_80_250', 'a_ICT2_80_250', 'a_ICT2_z_80_250', 'chan2roi_77bipolar', 'f', 'fmax_80_250', 'fmin_80_250', 'fs', 'x_ictal1_80_250', 'x_ictal2_80_250']


KeyError: 'x_preictal1_80_250'

In [ ]:
import re
from matplotlib.patches import Patch # per legenda manuale

# Input: states multi-banda da .mat
LUT_XLSX   = BASE / "centroidsMNI2mm_850ROIs.xlsx"
STATE_MATE = BASE / "within_phases_states.mat"
mat_states = loadmat(STATE_MATE)

# ROI IDs (per etichette)
b0_ids_vec = pd.read_csv(
    B0_TXT, sep=r"[\s,;]+", engine="python", header=None, comment="#"
).iloc[:, 0].astype(int).to_numpy()
assert len(b0_ids_vec) == N, "B0_TXT non coerente con N"

GROUP_COL = {
    # corticali
    "Vis": "silver",
    "SomMot": "yellow",
    "DorsAttn": "orange",
    "SalVentAttn": "red",
    "Limbic": "purple",
    "Cont": "blue",
    "Default": "green",
    # subcorticali
    "HIP": "brown",
    "THA": "cyan",
    "PUT": "magenta",
    "CAU": "lime",
    "AMY": "pink",
    "NAc": "teal",
    "GP": "gold",
}

CORT_ORDER = ["Vis","SomMot","DorsAttn","SalVentAttn","Limbic","Cont","Default"]
SUB_ORDER  = ["HIP","THA","PUT","CAU","AMY","NAc","GP"]

def cortical_network_from_roi_name(roi_name: str) -> str:
    if roi_name is None or (isinstance(roi_name, float) and np.isnan(roi_name)):
        return "Unknown"
    s = str(roi_name)
    for net in CORT_ORDER:
        if re.search(rf"(?:^|[_\s-]){re.escape(net)}(?:$|[_\s-])", s):
            return net
    return "Unknown"

def subcortex_group_from_roi_name(roi_name: str) -> str:
    if roi_name is None or (isinstance(roi_name, float) and np.isnan(roi_name)):
        return "Unknown"
    s = str(roi_name).replace("_", "-")
    s = re.sub(r"-(lh|rh)$", "", s, flags=re.IGNORECASE)

    if s.startswith("lAMY") or s.startswith("mAMY"):
        return "AMY"
    if s.startswith("pGP") or s.startswith("aGP"):
        return "GP"
    for pref in ("HIP", "THA", "PUT", "CAU", "NAc"):
        if s.startswith(pref + "-") or s == pref:
            return pref
    return "Unknown"

def group_from_roi(roi_id: int, roi_name: str) -> str:
    # 801..850 = subcortex (Tian)
    if int(roi_id) >= 801:
        g = subcortex_group_from_roi_name(roi_name)
        return g if g in SUB_ORDER else "Unknown"
    # altrimenti cortex
    g = cortical_network_from_roi_name(roi_name)
    if g in CORT_ORDER:
        return g
    # fallback subcortex (se naming strano)
    g2 = subcortex_group_from_roi_name(roi_name)
    return g2 if g2 in SUB_ORDER else "Unknown"

def build_roiid_to_group(lut_xlsx_path) -> dict:
    lut = pd.read_excel(lut_xlsx_path).rename(columns={
        "ROI Label": "roi_id",
        "ROI Name": "roi_name",
    })
    lut = lut.dropna(subset=["roi_id"]).copy()
    lut["roi_id"] = pd.to_numeric(lut["roi_id"], errors="coerce").astype(int)

    lut["group"] = lut.apply(lambda r: group_from_roi(r["roi_id"], r["roi_name"]), axis=1)
    return dict(zip(lut["roi_id"].astype(int).tolist(), lut["group"].tolist()))

roiid_to_group = build_roiid_to_group(LUT_XLSX)

def colors_and_legend_for_indices(node_indices, b0_ids_vec, roiid_to_group, group_col=GROUP_COL, unknown_color="lightgray"):
    """
    node_indices: indici 0-based dei nodi (es. top_all_abs)
    b0_ids_vec: vettore che mappa indice->roi_id
    """
    colors = []
    present = set()
    has_unknown = False
    for i in node_indices:
        roi_id = int(b0_ids_vec[i])
        g = roiid_to_group.get(roi_id, "Unknown")
        c = group_col.get(g, unknown_color)
        colors.append(c)
        if g == "Unknown":
            has_unknown = True
        else:
            present.add(g)

     # ordine legenda fisso (cortex poi subcortex)
    ordered_groups = [g for g in (CORT_ORDER + SUB_ORDER) if g in present]

    handles = [Patch(facecolor=group_col[g], edgecolor="black", label=g) for g in ordered_groups]
    if has_unknown:
        handles.append(Patch(facecolor=unknown_color, edgecolor="black", label="Unknown"))

    return colors, handles


# funzione di z-score
def zscore_1d(x: np.ndarray, ddof: int = 0) -> np.ndarray:
    """Standard z-score on a 1D array."""
    x = np.asarray(x, dtype=float)
    mu = np.nanmean(x)
    sd = np.nanstd(x, ddof=ddof)
    if not np.isfinite(sd) or sd <= 0:
        return np.full_like(x, np.nan, dtype=float)
    return (x - mu) / sd

# z-score within-band across nodes (850 ROI)
node_energy_z = zscore_1d(node_energy)


# =========================
# Plot: barplots energia per nodo (assoluta + z-score)
# (ordinati per valore decrescente TOP43)
# =========================

TOP_ALL = 75

order_all_abs = np.argsort(node_energy)[::-1]   # indici 0-based dei 850 nodi
top_all_abs = order_all_abs[:TOP_ALL]
colors, handles = colors_and_legend_for_indices(top_all_abs, b0_ids_vec, roiid_to_group)

# --- TOP 75 tra TUTTE le ROI per banda ---
plt.figure(figsize=(14, 4))
plt.bar(np.arange(TOP_ALL), node_energy[top_all_abs], color=colors)
plt.xticks(np.arange(TOP_ALL), [str(int(b0_ids_vec[i])) for i in top_all_abs], rotation=90)    
plt.xlabel("ROI ID (top 75 sorted by energy, all nodes)")
plt.ylabel("node control energy (absolute)")
plt.title(f"Top {TOP_ALL} node energies — [80 250] Hz (absolute, all ROIs sorted)")
plt.legend(handles=handles, title="Network / Subcortex", bbox_to_anchor=(1.01, 1), loc="upper left", frameon=True)
plt.tight_layout()
plt.savefig(RESULT / f"bar_top{TOP_ALL}_all_abs_[80 250].png", dpi=300, bbox_inches="tight")
plt.close()

# --- TOP 75 tra TUTTE le ROI per banda (z-score) ---
order_all_z = np.argsort(node_energy_z)[::-1]
top_all_z = order_all_z[:TOP_ALL]
colors, handles = colors_and_legend_for_indices(top_all_z, b0_ids_vec, roiid_to_group)

plt.figure(figsize=(14, 4))
plt.bar(np.arange(TOP_ALL), node_energy_z[top_all_z], color=colors)
plt.xticks(np.arange(TOP_ALL), [str(int(b0_ids_vec[i])) for i in top_all_z], rotation=90)
plt.xlabel("ROI ID (top 75 sorted by energy, all nodes)")
plt.ylabel("node control energy (z-score within band)")
plt.title(f"Top {TOP_ALL} node energies — [80 250] Hz (z-score within band, all ROIs sorted)")
plt.legend(handles=handles, title="Network / Subcortex", bbox_to_anchor=(1.01, 1), loc="upper left", frameon=True)
plt.tight_layout()
plt.savefig(RESULT / f"bar_top{TOP_ALL}_all_z_[80 250].png", dpi=300, bbox_inches="tight")
plt.close()


# Versione più leggibile con ROI ID in ascissa
TOP = 43
idx_ctrl = np.where(controllers)[0]   # indici 0-based dei controller (qui sono 43)

# --- TOP 43 CONTROLLERS per energia assoluta ---
order_ctrl_abs = idx_ctrl[np.argsort(node_energy[idx_ctrl])[::-1]]
top_abs = order_ctrl_abs[:TOP]
colors, handles = colors_and_legend_for_indices(top_abs, b0_ids_vec, roiid_to_group)
plt.figure(figsize=(12, 4))
plt.bar(np.arange(TOP), node_energy[top_abs], color=colors)
plt.xticks(np.arange(TOP), [str(int(b0_ids_vec[i])) for i in top_abs], rotation=90)
plt.xlabel("ROI ID (top 43)")
plt.ylabel("node control energy (absolute)")
plt.title(f"Top {TOP} node energies — [80 250] Hz (absolute)")
plt.legend(handles=handles, title="Network / Subcortex", bbox_to_anchor=(1.01, 1), loc="upper left", frameon=True)
plt.tight_layout()
plt.savefig(RESULT / f"bar_top{TOP}_abs_[80 250].png", dpi=300, bbox_inches="tight")
plt.close()

# --- TOP 43 CONTROLLERS per z-score calcolato sui soli controllers ---
E_ctrl = node_energy[idx_ctrl]   # energia assoluta dei controller
# z-score solo tra controller (non rispetto a tutti gli 850)
mu_c = E_ctrl.mean()
sd_c = E_ctrl.std(ddof=0)
E_ctrl_z = (E_ctrl - mu_c) / sd_c if sd_c > 0 else np.full_like(E_ctrl, np.nan)

TOPC = min(43, len(idx_ctrl))                       # nel mio caso 43
order_ctrl = np.argsort(E_ctrl_z)[::-1]             # ordino per z-score controller-only
idx_top_ctrl = idx_ctrl[order_ctrl[:TOPC]]          # indici globali (0-based)
colors, handles = colors_and_legend_for_indices(idx_top_ctrl, b0_ids_vec, roiid_to_group)


plt.figure(figsize=(12, 4))
plt.bar(np.arange(TOPC), E_ctrl_z[order_ctrl[:TOPC]], color=colors)  # uso la versione già ordinata
plt.xticks(np.arange(TOPC), [str(int(b0_ids_vec[i])) for i in idx_top_ctrl], rotation=90)
plt.xlabel("ROI ID (top controllers)")
plt.ylabel("z-score (within controllers only)")
plt.title(f"Top {TOPC} controllers — [80 250] Hz (z-score within controllers)")
plt.legend(handles=handles, title="Network / Subcortex", bbox_to_anchor=(1.01, 1), loc="upper left", frameon=True)
plt.tight_layout()
plt.savefig(RESULT / f"bar_top{TOPC}_controllers_z_[80 250].png", dpi=300, bbox_inches="tight")
plt.close()

# --- TOP 43 CONTROLLERS per z-score (z-score calcolato su tutti gli 850 nodi) ---
order_ctrl_z = idx_ctrl[np.argsort(node_energy_z[idx_ctrl])[::-1]]
top_z = order_ctrl_z[:TOP]
colors, handles = colors_and_legend_for_indices(top_z, b0_ids_vec, roiid_to_group)
plt.figure(figsize=(12, 4))
plt.bar(np.arange(TOP), node_energy_z[top_z], color=colors)
plt.xticks(np.arange(TOP), [str(int(b0_ids_vec[i])) for i in top_z], rotation=90)
plt.xlabel("ROI ID (top 43)")
plt.ylabel("node control energy (z-score within band)")
plt.title(f"Top {TOP} node energies — [80 250] Hz (z-score within band)")
plt.legend(handles=handles, title="Network / Subcortex", bbox_to_anchor=(1.01, 1), loc="upper left", frameon=True)
plt.tight_layout()
plt.savefig(RESULT / f"bar_top{TOP}_z_[80 250].png", dpi=300, bbox_inches="tight")
plt.close()


CHECK COERENZA ROI <-> EMISFERO, ipsi/contra

In [ ]:
# 1) costruisco una LUT (roi_id -> roi_name, mni_x/y/z) da centroidsMNI2mm_850ROIs.xlsx
# 2) controllo coerenza emisfero: da roi_name (se contiene RH/Lh) e da coord MNI (segno x < 0 sinistra, viceversa destra)
# 3) creo colonna ipsi/contra in base al lato del mio focus (LTE sx)
# 4) per ogni banda, unisco LUT + top controllers e produco
#     - tabella top-50 controllers con rank_roi, roi_name, hemi_name, hemi_mni, ipsi_contra, node_energy
#     - summary per banda: quante ROI in ipsi/contra, quanta energia in ipsi/contra e mismatch L/R


# PARAMETRI
FOCUS_SIDE = "L"  # "L" se TLE sinistra, "R" se TLE destra
TOPK = 43         # top controller da analizzare per banda

RESULT_DIR = OUT / "control_energy_PREICTAL"  # dove ho salvato top_nodes_*.csv
EXPORT_DIR = RESULT_DIR / f"exports_top{TOPK}" 
EXPORT_DIR.mkdir(parents=True, exist_ok=True) 

# ==========================
# FUNZIONI: parse ROI name
# ==========================
def parse_hemi(roi_name: str):
    if roi_name is None or (isinstance(roi_name, float) and np.isnan(roi_name)):
        return np.nan
    s = str(roi_name)
    m = re.search(r"_(LH|RH)_", s) # caso corticale
    if m:
        return m.group(1)
    m = re.search(r"-(lh|rh)(?:$|[_\-\s])", s, flags=re.IGNORECASE) # caso subcorticale
    if m:
        return m.group(1).upper() 
    
    return np.nan

def parse_network(roi_name: str):
    if roi_name is None or (isinstance(roi_name, float) and np.isnan(roi_name)):
        return np.nan
    s = str(roi_name)
    # subcortex: nome tipo HIP-head-lh
    if re.search(r"-(lh|rh)(?:$|[_\-\s])", s, flags=re.IGNORECASE):
        return "Subcortex"
    # es: 7Networks_LH_Default_pCunPCC_27 -> network=Default
    toks = s.split("_")
    # pattern atteso: [ '7Networks', 'LH', 'Default', ...]
    if len(toks) >= 3 and toks[1] in ("LH", "RH"):
        return toks[2]
    return np.nan

def ipsi_contra_from_hemi(hemi: str):
    if hemi not in ("LH", "RH"):
        return np.nan
    # TLE sinistra => ipsi=LH
    if FOCUS_SIDE.upper() == "L":
        return "ipsi" if hemi == "LH" else "contra"
    else:
        return "ipsi" if hemi == "RH" else "contra"


# =========================
# 1) LUT (ROI ID -> ROI name))
# =========================
# carico LUT (centroidsMNI2mm_850ROIs.xlsx)
lut = pd.read_excel(MNI_XLSX).rename(columns={
    "ROI Label": "roi_id",
    "ROI Name": "roi_name",
})

# pulizia tipi
lut["roi_id"] = pd.to_numeric(lut["roi_id"], errors="coerce").astype("Int64")
lut = lut.dropna(subset=["roi_id"]).copy()
lut["roi_id"] = lut["roi_id"].astype(int)

# estraggo emisfero da nome e da coordinate
lut["hemi"] = lut["roi_name"].apply(parse_hemi)
lut["network"] = lut["roi_name"].apply(parse_network)
lut["ipsi_contra"] = lut["hemi"].apply(ipsi_contra_from_hemi)

# salvo lookup “pulita”
lookup_cols = ["roi_id","roi_name","hemi","network","ipsi_contra"]
lut_lookup = lut[lookup_cols].copy()
lut_lookup.to_csv(EXPORT_DIR / "roi_lookup_850_named.csv", index=False)
lut_lookup.to_excel(EXPORT_DIR / "roi_lookup_850_named.xlsx", index=False, engine="openpyxl")

print("[OK] LUT pronta:", lut_lookup.shape, "| salvata in", EXPORT_DIR)

# =========================
# 2) Analisi: top43 + LH/RH + summary ipsi/contra
# =========================
summary_rows = []

for band in BANDS_LIST:
    band_dir = RESULT_DIR / band
    node_path = band_dir / f"node_energy_850_{band}.csv"  # <-- NUOVO input per banda
    if not node_path.exists():
        print(f"[SKIP] Non trovo {node_path}")
        continue

    df = pd.read_csv(node_path)
    need = {"roi_id", "node_energy", "is_controller"}
    if not need.issubset(df.columns):
        print(f"[SKIP] {node_path.name} colonne mancanti. Trovate: {list(df.columns)}")
        continue


    # ---- top SOLO controller (is_controller=1), ordinati per energia assoluta ----
    ctrl = df[df["is_controller"].astype(int) == 1].copy()
    ctrl = ctrl.sort_values("node_energy", ascending=False).head(TOPK).reset_index(drop=True)
    ctrl["rank"] = np.arange(1, len(ctrl) + 1)

    # merge con LUT
    ctrl = ctrl.merge(lut_lookup, on="roi_id", how="left")

    # tabella finale top controllers
    ctrl_named = ctrl[["rank", "roi_id", "roi_name", "hemi", "network", "ipsi_contra", "node_energy"]].copy()

    # split LH/RH
    ctrl_LH = ctrl_named[ctrl_named["hemi"] == "LH"].copy()
    ctrl_RH = ctrl_named[ctrl_named["hemi"] == "RH"].copy()

    # (opzionale) warning per ROI non matchate nella LUT
    n_missing = int(ctrl_named["hemi"].isna().sum())
    if n_missing > 0:
        print(f"[WARN] Banda {band}: {n_missing}/{len(ctrl_named)} top ROI senza hemi (merge LUT mancante?)")

    # summary ipsi/contra
    E_total  = float(ctrl_named["node_energy"].sum())
    E_ipsi   = float(ctrl_named.loc[ctrl_named["ipsi_contra"] == "ipsi", "node_energy"].sum())
    E_contra = float(ctrl_named.loc[ctrl_named["ipsi_contra"] == "contra", "node_energy"].sum())
    n_ipsi   = int((ctrl_named["ipsi_contra"] == "ipsi").sum())
    n_contra = int((ctrl_named["ipsi_contra"] == "contra").sum())

    summary_rows.append({
        "band": band,
        f"top{TOPK}_E_total": E_total,
        f"top{TOPK}_E_ipsi": E_ipsi,
        f"top{TOPK}_E_contra": E_contra,
        f"top{TOPK}_n_ipsi": n_ipsi,
        f"top{TOPK}_n_contra": n_contra,
    })

    # export per banda
    out_band_dir = EXPORT_DIR / f"band_{band}"
    out_band_dir.mkdir(parents=True, exist_ok=True)

    ctrl_named.to_csv(out_band_dir / f"top{TOPK}_controllers_{band}_named.csv", index=False)
    ctrl_LH.to_csv(out_band_dir / f"top{TOPK}_controllers_{band}_LH.csv", index=False)
    ctrl_RH.to_csv(out_band_dir / f"top{TOPK}_controllers_{band}_RH.csv", index=False)

    with pd.ExcelWriter(out_band_dir / f"top{TOPK}_controllers_{band}.xlsx", engine="openpyxl") as w:
        ctrl_named.to_excel(w, sheet_name=f"top{TOPK}", index=False)
        ctrl_LH.to_excel(w, sheet_name="LH", index=False)
        ctrl_RH.to_excel(w, sheet_name="RH", index=False)

    print(f"[OK] Banda {band}: salvati CSV+XLSX in {out_band_dir}")

# export summary cross-banda
summary_df = pd.DataFrame(summary_rows).sort_values("band")
summary_df.to_csv(EXPORT_DIR / f"summary_ipsi_contra_top{TOPK}.csv", index=False)
summary_df.to_excel(EXPORT_DIR / f"summary_ipsi_contra_top{TOPK}.xlsx", index=False, engine="openpyxl")

print("\n=== SUMMARY ipsi/contra (top controllers) ===")
print(summary_df.to_string(index=False))
print("\n[OK] Salvato summary in:", EXPORT_DIR)